In [1]:
from google.colab import drive
drive.mount('/content/drive')
filepath ="/content/drive/MyDrive/Colab Notebooks/PGDBA-Project/V2"

Mounted at /content/drive


spitting into train and test folder

In [ ]:
import splitfolders

splitfolders.ratio("E:\\Audio-Classification-master\\Audio-Classification-master\\wavfiles", output="E:\\Audio-Classification-master\\Audio-Classification-master\\wavfiles_split",
    seed=1337, ratio=(.9, .1), group_prefix=None, move=False)

making CSV with fname and label

In [ ]:
import os
import csv

desktop_path = os.path.expanduser("E:\Audio-Classification-master\Audio-Classification-master\wavfiles_split")
main_folder = "test"

# Create the CSV file and write the header row
csv_file_path = os.path.join(desktop_path, "file_folder.csv")
with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["File Name", "Folder Name"])

    # Iterate over the subfolders and files
    for folder_name in os.listdir(os.path.join(desktop_path, main_folder)):
        folder_path = os.path.join(desktop_path, main_folder, folder_name)
        if os.path.isdir(folder_path):
            for file_name in os.listdir(folder_path):
                writer.writerow([file_name, folder_name])

print("CSV file has been created successfully at:", csv_file_path)

merging all subfolders

In [ ]:
import os
import shutil

main_folder_path = "E:\\Audio-Classification-master\\Audio-Classification-master\\wavfiles_split\\test"

# Get the list of subfolders in the main folder
subfolders = [f.path for f in os.scandir(main_folder_path) if f.is_dir()]

# Move files from each subfolder to the main folder
for subfolder_path in subfolders:
    files = os.listdir(subfolder_path)
    for file in files:
        file_path = os.path.join(subfolder_path, file)
        shutil.move(file_path, main_folder_path)

cleaning audio files

In [2]:

from tqdm import tqdm
import pandas as pd
import numpy as np
import librosa
from scipy.io import wavfile

In [3]:
def envelope(y,rate,threshold):
  mask=[]
  y=pd.Series(y).apply(np.abs)
  y_mean=y.rolling(window=int(rate/10),min_periods=1,center=True).mean()
  for mean in y_mean:
    if(mean>threshold):
      mask.append(True)
    else:
      mask.append(False)
  return(mask)

In [7]:
df=pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PGDBA-Project/V2/train_inst.csv")

In [10]:
for f in tqdm(df.fname):
  signal,rate=librosa.load("/content/drive/MyDrive/Colab Notebooks/PGDBA-Project/V2/train/"+f,sr=16000)
  mask=envelope(signal,rate,0.0005)
  wavfile.write(filename="/content/drive/MyDrive/Colab Notebooks/PGDBA-Project/V2/clean_train/"+f,rate=rate,data=signal[mask])

100%|██████████| 270/270 [00:12<00:00, 22.19it/s]


In [11]:
df=pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PGDBA-Project/V2/test_inst.csv")

for f in tqdm(df.fname):
  signal,rate=librosa.load("/content/drive/MyDrive/Colab Notebooks/PGDBA-Project/V2/test/"+f,sr=16000)
  mask=envelope(signal,rate,0.0005)
  wavfile.write(filename="/content/drive/MyDrive/Colab Notebooks/PGDBA-Project/V2/clean_test/"+f,rate=rate,data=signal[mask])

100%|██████████| 30/30 [00:08<00:00,  3.65it/s]
